In [1]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
from tqdm import tqdm
import torch
import tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
embedding_model_name = "all-MiniLM-L6-v2"
digits = 6

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16, device_map="auto")

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
reviews = pd.read_pickle("data/data_processed/review_clean.pkl")

In [5]:
# extract documents and convert to JSON lines format
docs = pd.read_pickle("data/data_processed/naics_clean_{}.pkl".format(digits))

In [20]:
data_naics = []
docs = pd.read_pickle("data/data_processed/naics_clean_{}.pkl".format(digits))
for naics in reviews["naics_code_1_mapped_2022"].astype(str):
    if naics in docs["naics"].values :
        data_naics.append(naics)

In [7]:
def prompt_generator(naics, doc):
    title = docs.loc[doc["naics"] == naics, "title"].values[0][0]
    description = docs.loc[doc["naics"] == naics, "description"].values[0]
    
    prompt = f"""
Please generate synthetic business information of business content and categories 
using the North American Industry Classification System (NAICS) 
based on the following NAICS information provided by the business. 

The business content is a long paragraph of sentences that describe the business's scope/coverage.
The business categories are a short list of words that the business used to categorize themselves.

The business's naics code: {naics}
The naics's title: {title}
The naics's description: {description}

Please parse "answer" and output them in JSON format.

EXAMPLE JSON OUTPUT:
{{
    "content": [CONTENT],
    "categories": [CATEGORIES],
}}
"""
    
    return prompt

def get_llm_response(naics, doc, max_len: int = 2048, temperature = 0.0001) :
    
    def post_processing(generated_token_ids) :
        return generated_token_ids[:, len(model_inputs["input_ids"][0]):]

    prompt = prompt_generator(naics, doc)
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant to generate synthetic data for insurance industry classification."},
        {"role": "user", "content": prompt}
    ]
    
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([input_text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_len,
        pad_token_id=tokenizer.eos_token_id,
        temperature=temperature,
    )
    generated_token_ids_post_processed = post_processing(generated_ids)
    
    response = tokenizer.batch_decode(generated_token_ids_post_processed, skip_special_tokens=True)[0]
    
    return response

In [ ]:
import re, json

def extract(text) :
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        data = json.loads(match.group(0))
    return data

def generate_info(naics, doc) :
    try:
        info = extract(get_llm_response(naics, doc))
        if naics in info["content"] :
            raise ValueError
        return {"content": info["content"], "categories": info["categories"]}
    except:
        return generate_info(naics, doc)

data = {"naics": [], "content": [], "categories": []}
for naics in tqdm(data_naics):
    data["naics"].append(naics)
    info = generate_info(naics, docs)
    data["content"].append(info["content"]) 
    data["categories"].append(info["categories"]) 
    # pd.DataFrame(data).to_csv("llm_syn_review.csv")

In [12]:
import pandas as pd
data = pd.read_csv("llm_syn_review.csv")
data

,naics,content,categories
0,325180,XYZ Chemicals is a leading manufacturer specia...,"['Chemical Manufacturing', 'Inorganic Chemical..."
1,325613,This business specializes in the manufacturing...,"['surface active agents', 'emulsifiers', 'text..."
2,923120,This organization is dedicated to the administ...,"['Public Health Administration', 'Cancer Detec..."
3,323120,The business specializes in support printing s...,"['Bookbinding', 'Prepress Printing', 'Plate Pr..."
4,311920,Our company specializes in the comprehensive m...,"['Coffee Manufacturing', 'Tea Manufacturing', ..."
...,...,...,...
2367,325312,This business specializes in the manufacturing...,"['phosphatic fertilizer manufacturing', 'diamm..."
2368,813319,This organization is dedicated to promoting so...,"['social advocacy', 'accident prevention', 'an..."
2369,541320,Our firm is a professional landscape architect...,"['landscape architecture', 'urban planning', '..."
2370,813319,This organization is dedicated to promoting so...,"['advocacy', 'social change', 'human rights', ..."
